# 03 - Embeddings de productos

Este notebook cubre la etapa de similitud de productos. Entreno embeddings basados en co-ocurrencia de productos dentro de órdenes y también embeddings textuales para búsquedas por palabra o frase.

La parte de co-ocurrencia sirve para `get_similar_products(product_id)`. La parte textual sirve para `search_by_keyword(query)`.


## Instalación de librerías

Instalo las librerías necesarias para embeddings y búsqueda vectorial. En Kaggle cada notebook tiene su propio entorno, por eso esta instalación se deja explícita.


In [2]:
!pip install -q gensim sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 68.5 MB/s eta 0:00:00


## Localización de inputs

Busco los Parquet procesados del notebook `01_prepare_data`. Para esta etapa solo necesito el historial `prior_items` y el catálogo de productos.


In [3]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Buscar automáticamente la carpeta que contiene los parquets procesados
prior_files = list(INPUT.rglob("prior_items.parquet"))

if not prior_files:
    raise FileNotFoundError(
        "No se encontró prior_items.parquet. "
        "Agrega como input el output del notebook 01_prepare_data."
    )

DATA_DIR = prior_files[0].parent

print("Datos procesados encontrados en:")
print(DATA_DIR)

print("\nArchivos disponibles:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Datos procesados encontrados en:
/kaggle/input/notebooks/samanthacarolina/01-prepare-data

Archivos disponibles:
- __results__.html
- product_catalog.parquet
- __notebook__.ipynb
- orders.parquet
- __output__.json
- train_items.parquet
- prior_items.parquet
- custom.css


## Configuración del entorno

Dejo juntos los imports y las rutas de trabajo. Esta celda repite parte de la configuración para que el notebook pueda ejecutarse de forma independiente.


In [4]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Buscar automáticamente la carpeta que contiene los parquets procesados
prior_files = list(INPUT.rglob("prior_items.parquet"))

if not prior_files:
    raise FileNotFoundError(
        "No se encontró prior_items.parquet. "
        "Agrega como input el output del notebook 01_prepare_data."
    )

DATA_DIR = prior_files[0].parent

print("Datos procesados encontrados en:")
print(DATA_DIR)

print("\nArchivos disponibles:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Datos procesados encontrados en:
/kaggle/input/notebooks/samanthacarolina/01-prepare-data

Archivos disponibles:
- __results__.html
- product_catalog.parquet
- __notebook__.ipynb
- orders.parquet
- __output__.json
- train_items.parquet
- prior_items.parquet
- custom.css


## Carga mínima de datos

Cargo únicamente las columnas necesarias para embeddings. Esto ayuda a reducir memoria y tiempo de ejecución.


In [5]:
prior_items = pd.read_parquet(
    DATA_DIR / "prior_items.parquet",
    columns=["order_id", "product_id", "add_to_cart_order"]
)

product_catalog = pd.read_parquet(DATA_DIR / "product_catalog.parquet")

print("prior_items:", prior_items.shape)
print("product_catalog:", product_catalog.shape)

prior_items.head()

prior_items: (32434489, 3)
product_catalog: (49688, 6)


,order_id,product_id,add_to_cart_order
0,2,33120,1
1,2,28985,2
2,2,9327,3
3,2,45918,4
4,2,30035,5


## Preparación del catálogo

Creo un diccionario de búsqueda por `product_id` para poder mostrar nombres, aisle y department en los resultados.


In [6]:
catalog_min = product_catalog[
    [
        "product_id",
        "product_name",
        "aisle_id",
        "aisle",
        "department_id",
        "department"
    ]
].drop_duplicates("product_id").copy()

catalog_min["product_id"] = catalog_min["product_id"].astype(int)

product_lookup = (
    catalog_min
    .set_index("product_id")
    .to_dict(orient="index")
)

print("Productos en catálogo:", len(product_lookup))
catalog_min.head()

Productos en catálogo: 49688


,product_id,product_name,aisle_id,aisle,department_id,department
0,1,Chocolate Sandwich Cookies,61,cookies cakes,19,snacks
1,2,All-Seasons Salt,104,spices seasonings,13,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,tea,7,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,frozen meals,1,frozen
4,5,Green Chile Anytime Sauce,5,marinades meat preparation,13,pantry


## Baskets para Item2Vec

Cada orden se trata como una canasta de productos. La idea es que productos que aparecen juntos con frecuencia terminen cerca en el espacio vectorial.


In [7]:
# Puedes limitar para una primera corrida rápida.
# Si quieres usar todo, pon MAX_BASKETS = None.
MAX_BASKETS = 250_000

if MAX_BASKETS is not None:
    sampled_orders = (
        prior_items["order_id"]
        .drop_duplicates()
        .sample(
            n=min(MAX_BASKETS, prior_items["order_id"].nunique()),
            random_state=42
        )
    )

    prior_for_embeddings = prior_items[
        prior_items["order_id"].isin(sampled_orders)
    ].copy()
else:
    prior_for_embeddings = prior_items.copy()

prior_for_embeddings = prior_for_embeddings.sort_values(
    ["order_id", "add_to_cart_order"]
)

baskets = (
    prior_for_embeddings
    .groupby("order_id")["product_id"]
    .apply(lambda x: [str(product_id) for product_id in x.tolist()])
    .tolist()
)

print("Cantidad de baskets:", len(baskets))
print("Ejemplo de basket:")
print(baskets[0][:20])

Cantidad de baskets: 250000
Ejemplo de basket:
['20392', '27845', '162', '2452', '8575', '41890', '39475', '10096', '23032', '20995', '45066']


## Entrenamiento de Item2Vec

Uso Word2Vec sobre las canastas. En este contexto cada producto funciona como una palabra y cada orden como una frase.


In [8]:
from gensim.models import Word2Vec

item2vec = Word2Vec(
    sentences=baskets,
    vector_size=128,
    window=20,
    min_count=2,
    sg=1,
    negative=10,
    epochs=5,
    workers=4,
    seed=42
)

item2vec.save(str(OUTPUT_DIR / "item2vec.model"))

print("Modelo Item2Vec entrenado y guardado.")
print("Productos con embedding:", len(item2vec.wv))

Modelo Item2Vec entrenado y guardado.
Productos con embedding: 35414


## Función `get_similar_products`

Esta función consulta el modelo Item2Vec y devuelve productos similares con su score. También agrega metadatos del catálogo para que el resultado sea legible.


In [9]:
def get_similar_products(product_id: int, k: int = 10):
    """
    Retorna los K productos más similares usando Item2Vec.
    La similitud viene de co-ocurrencia en órdenes.
    """
    product_id = int(product_id)
    key = str(product_id)

    if key not in item2vec.wv:
        return {
            "query_product_id": product_id,
            "error": "product_id no tiene embedding Item2Vec",
            "results": []
        }

    query_info = product_lookup.get(product_id, {})

    similar_items = item2vec.wv.most_similar(key, topn=k + 5)

    results = []

    for similar_product_id, score in similar_items:
        similar_product_id = int(similar_product_id)

        # Evitar retornar el mismo producto
        if similar_product_id == product_id:
            continue

        info = product_lookup.get(similar_product_id)

        if info is None:
            continue

        results.append({
            "product_id": similar_product_id,
            "product_name": info["product_name"],
            "aisle": info["aisle"],
            "department": info["department"],
            "score": float(score)
        })

        if len(results) >= k:
            break

    return {
        "query_product_id": product_id,
        "query_product_name": query_info.get("product_name", None),
        "results": results
    }

## Prueba con Banana

Hago una prueba rápida con un producto común para revisar si los resultados tienen sentido.


In [10]:
example_product_id = 24852

similar_results = get_similar_products(example_product_id, k=10)

print("Producto consultado:")
print(similar_results["query_product_id"], "-", similar_results["query_product_name"])

print("\nProductos similares:")
pd.DataFrame(similar_results["results"])

Producto consultado:
24852 - Banana

Productos similares:


,product_id,product_name,aisle,department,score
0,13176,Bag of Organic Bananas,fresh fruits,produce,0.688198
1,15754,Dark Chocolate High Protein Bars,protein meal replacements,personal care,0.659177
2,48892,Hunger Filler Whole grain Bread,bread,bakery,0.653852
3,42390,Organic Apple Gala Bag,packaged vegetables fruits,produce,0.652038
4,48327,Organic Kale Caesar Salad Kit,packaged vegetables fruits,produce,0.650143
5,24275,Golean Vanilla Graham Clusters Cereal,cereal,breakfast,0.649253
6,20029,Apple Chardonnay Chicken Sausage,hot dogs bacon sausage,meat seafood,0.648145
7,39365,French Vanilla Syrup,honeys syrups nectars,pantry,0.647208
8,14883,Organic Cream Top Vanilla Yogurt,yogurt,dairy eggs,0.646705
9,19822,Kosher Red Apple Havarti Shingles,kosher foods,international,0.646580


## Segunda prueba de similitud

Uso otro producto popular como sanity check adicional.


In [11]:
# Organic Strawberries suele estar en el catálogo como producto popular
get_similar_products(21137, k=10)

{'query_product_id': 21137,
 'query_product_name': 'Organic Strawberries',
 'results': [{'product_id': 3640,
   'product_name': 'Caesar Rules Dressing',
   'aisle': 'salad dressing toppings',
   'department': 'pantry',
   'score': 0.7619569897651672},
  {'product_id': 2578,
   'product_name': 'Whole Grain Pancakes',
   'aisle': 'frozen breakfast',
   'department': 'frozen',
   'score': 0.759232223033905},
  {'product_id': 21298,
   'product_name': 'Organic Cake Cones',
   'aisle': 'ice cream ice',
   'department': 'frozen',
   'score': 0.7532902956008911},
  {'product_id': 17118,
   'product_name': 'Natural Sugar Free Spearmint Mints',
   'aisle': 'mint gum',
   'department': 'snacks',
   'score': 0.751467227935791},
  {'product_id': 32436,
   'product_name': 'Original Granola',
   'aisle': 'granola',
   'department': 'breakfast',
   'score': 0.7506614327430725},
  {'product_id': 41162,
   'product_name': 'Grapes, Certified Organic, California, Black Seedless',
   'aisle': 'packaged ve

## Texto para embeddings semánticos

Además de co-ocurrencia, creo una descripción corta por producto usando nombre, aisle y departamento. Esto permite buscar por palabras como `milk` o frases como `healthy breakfast`.


In [12]:
texts_df = catalog_min.copy()

texts_df["text_for_embedding"] = (
    texts_df["product_name"].astype(str)
    + " | "
    + texts_df["aisle"].astype(str)
    + " | "
    + texts_df["department"].astype(str)
)

texts = texts_df["text_for_embedding"].tolist()

print("Cantidad de textos:", len(texts))
print(texts[:5])

Cantidad de textos: 49688
['Chocolate Sandwich Cookies | cookies cakes | snacks', 'All-Seasons Salt | spices seasonings | pantry', 'Robust Golden Unsweetened Oolong Tea | tea | beverages', 'Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce | frozen meals | frozen', 'Green Chile Anytime Sauce | marinades meat preparation | pantry']


## Generación de embeddings textuales

Uso `all-MiniLM-L6-v2` para obtener un vector por producto. Los vectores se normalizan para usar similitud coseno con FAISS.


In [13]:
from sentence_transformers import SentenceTransformer

text_model_name = "all-MiniLM-L6-v2"

text_model = SentenceTransformer(text_model_name)

text_embeddings = text_model.encode(
    texts,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True
)

text_embeddings = text_embeddings.astype("float32")

print("Shape embeddings:", text_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/195 [00:00<?, ?it/s]

Shape embeddings: (49688, 384)


## Guardado de vectores y mapping

Guardo los embeddings y el arreglo de product IDs alineado con las filas del vector. Esto es necesario para recuperar el producto después de una búsqueda.


In [14]:
np.save(OUTPUT_DIR / "text_embeddings.npy", text_embeddings)

product_ids = texts_df["product_id"].astype(int).to_numpy()
np.save(OUTPUT_DIR / "text_product_ids.npy", product_ids)

texts_df.to_parquet(OUTPUT_DIR / "product_catalog_embeddings.parquet", index=False)

print("Guardado:")
print("- text_embeddings.npy")
print("- text_product_ids.npy")
print("- product_catalog_embeddings.parquet")

Guardado:
- text_embeddings.npy
- text_product_ids.npy
- product_catalog_embeddings.parquet


## Índice FAISS

Creo un índice vectorial para que las búsquedas por keyword sean rápidas.


In [15]:
import faiss

embedding_dim = text_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(text_embeddings)

faiss.write_index(faiss_index, str(OUTPUT_DIR / "text_products.faiss"))

print("Índice FAISS creado.")
print("Cantidad de vectores:", faiss_index.ntotal)

Índice FAISS creado.
Cantidad de vectores: 49688


## Función `search_by_keyword`

Esta función genera el embedding de una palabra o frase y devuelve los productos más cercanos en el índice FAISS.


In [16]:
def search_by_keyword(query: str, k: int = 10):
    """
    Busca los K productos más cercanos semánticamente a una palabra o frase.
    """
    if not query or not query.strip():
        return []

    query_embedding = text_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        row = texts_df.iloc[int(idx)]

        results.append({
            "product_id": int(row["product_id"]),
            "product_name": row["product_name"],
            "aisle": row["aisle"],
            "department": row["department"],
            "score": float(score)
        })

    return results

## Prueba de búsqueda semántica

Pruebo una intención general de desayuno saludable para revisar si los productos recuperados son razonables.


In [17]:
keyword_results = search_by_keyword("healthy breakfast", k=10)

pd.DataFrame(keyword_results)

,product_id,product_name,aisle,department,score
0,22337,Great Grains,cereal,breakfast,0.664719
1,25808,English Breakfast,tea,beverages,0.637785
2,4455,Raspberry Cereal,breakfast bars pastries,breakfast,0.624243
3,19553,Raw Breakfast Crusts,breakfast bars pastries,breakfast,0.606130
4,40451,Breakfast Pack,cereal,breakfast,0.602548
5,15477,Wheaties,cereal,breakfast,0.600717
6,8243,Original Toastable Organic Breakfast,frozen breakfast,frozen,0.598905
7,13042,with Crispy Almonds Cereal,cereal,breakfast,0.593649
8,42446,Shredded Wheat,cereal,breakfast,0.592957
9,38601,Breakfast Americana,tea,beverages,0.587252


## Prueba con `milk`

Esta consulta ayuda a revisar si la búsqueda textual captura productos relacionados con leche o lácteos.


In [18]:
pd.DataFrame(search_by_keyword("milk", k=10))

,product_id,product_name,aisle,department,score
0,16611,Milk,milk,dairy eggs,0.679701
1,36524,Chocolate Drink,milk,dairy eggs,0.656180
2,41488,Chocolate Milk,milk,dairy eggs,0.643238
3,2187,Strawberry Milk,milk,dairy eggs,0.611151
4,15254,Milk of Magnesia,digestion,personal care,0.608596
5,4210,Whole Milk,milk,dairy eggs,0.607878
6,34197,Goat Milk,milk,dairy eggs,0.604648
7,19348,Fat Free Milk,milk,dairy eggs,0.596744
8,4879,"Milk, Fat Free",milk,dairy eggs,0.593248
9,45468,Lowfat Chocolate Milk,milk,dairy eggs,0.592484


## Prueba con `greek yogurt`

Otra consulta puntual para revisar coherencia en resultados de una categoría específica.


In [19]:
pd.DataFrame(search_by_keyword("greek yogurt", k=10))

,product_id,product_name,aisle,department,score
0,45041,Greek-Style Plain Yogurt,yogurt,dairy eggs,0.824910
1,45675,Greek Yogurt,yogurt,dairy eggs,0.815001
2,4636,Greek Style Honey Yogurt,other,other,0.809547
3,7156,Traditional Plain Greek Yogurt,yogurt,dairy eggs,0.807389
4,11422,Plain Greek Yogurt,yogurt,dairy eggs,0.806258
5,2068,Greek Strawberry Yogurt,yogurt,dairy eggs,0.800249
6,45848,Greek Yogurt Indulgent Raspberry,yogurt,dairy eggs,0.795728
7,11767,Greek Vanilla Yogurt,yogurt,dairy eggs,0.793389
8,9969,Raspberry Greek Yogurt,yogurt,dairy eggs,0.792455
9,13698,Greek Raspberry Whole Milk Yogurt,yogurt,dairy eggs,0.790922


## Medición de latencia

Mido varias consultas para verificar que el p95 esté por debajo de 2 segundos, como pide la prueba.


In [20]:
test_queries = [
    "milk",
    "healthy breakfast",
    "banana",
    "greek yogurt",
    "coffee",
    "oatmeal",
    "protein snack",
    "fresh fruit",
    "vegetables",
    "organic juice"
]

# Warmup
for q in test_queries[:3]:
    _ = search_by_keyword(q, k=10)

times = []

for q in test_queries * 10:
    start = time.perf_counter()
    _ = search_by_keyword(q, k=10)
    elapsed = time.perf_counter() - start
    times.append(elapsed)

latency_report = {
    "num_queries": len(times),
    "mean_ms": float(np.mean(times) * 1000),
    "p50_ms": float(np.percentile(times, 50) * 1000),
    "p95_ms": float(np.percentile(times, 95) * 1000),
    "max_ms": float(np.max(times) * 1000),
    "requirement_ms": 2000,
    "passes_requirement": bool(np.percentile(times, 95) * 1000 < 2000)
}

latency_report

{'num_queries': 100,
 'mean_ms': 20.387595179994378,
 'p50_ms': 20.09600099995623,
 'p95_ms': 22.60019175004686,
 'max_ms': 24.77529600002981,
 'requirement_ms': 2000,
 'passes_requirement': True}

## Ejemplos para la demo

Guardo algunos ejemplos de productos similares y búsquedas por keyword. La app de Streamlit puede mostrarlos sin recalcular embeddings.


In [21]:
embedding_examples = {
    "similar_products_examples": {
        "banana_24852": get_similar_products(24852, k=10),
        "organic_strawberries_21137": get_similar_products(21137, k=10),
    },
    "keyword_search_examples": {
        "healthy breakfast": search_by_keyword("healthy breakfast", k=10),
        "milk": search_by_keyword("milk", k=10),
        "coffee": search_by_keyword("coffee", k=10),
    }
}

with open(OUTPUT_DIR / "embedding_examples.json", "w") as f:
    json.dump(embedding_examples, f, indent=2)

with open(OUTPUT_DIR / "embedding_latency.json", "w") as f:
    json.dump(latency_report, f, indent=2)

print("Guardado:")
print("- embedding_examples.json")
print("- embedding_latency.json")

Guardado:
- embedding_examples.json
- embedding_latency.json


## Reporte general

Dejo un resumen de los parámetros usados, cantidad de productos con embedding y latencia obtenida.


In [22]:
embedding_report = {
    "item2vec": {
        "method": "Word2Vec over product baskets",
        "vector_size": 128,
        "window": 20,
        "min_count": 2,
        "epochs": 5,
        "max_baskets": MAX_BASKETS,
        "num_baskets": len(baskets),
        "num_products_with_embedding": len(item2vec.wv)
    },
    "text_embeddings": {
        "method": "SentenceTransformer over product_name + aisle + department",
        "model_name": text_model_name,
        "embedding_shape": list(text_embeddings.shape),
        "faiss_index": "IndexFlatIP over normalized embeddings"
    },
    "latency": latency_report
}

with open(OUTPUT_DIR / "embedding_report.json", "w") as f:
    json.dump(embedding_report, f, indent=2)

print(json.dumps(embedding_report, indent=2))

{
  "item2vec": {
    "method": "Word2Vec over product baskets",
    "vector_size": 128,
    "window": 20,
    "min_count": 2,
    "epochs": 5,
    "max_baskets": 250000,
    "num_baskets": 250000,
    "num_products_with_embedding": 35414
  },
  "text_embeddings": {
    "method": "SentenceTransformer over product_name + aisle + department",
    "model_name": "all-MiniLM-L6-v2",
    "embedding_shape": [
      49688,
      384
    ],
    "faiss_index": "IndexFlatIP over normalized embeddings"
  },
  "latency": {
    "num_queries": 100,
    "mean_ms": 20.387595179994378,
    "p50_ms": 20.09600099995623,
    "p95_ms": 22.60019175004686,
    "max_ms": 24.77529600002981,
    "requirement_ms": 2000,
    "passes_requirement": true
  }
}


## ZIP de artefactos de embeddings

Empaquete los archivos principales de embeddings para poder reutilizarlos en el notebook de exportación.


In [23]:
import shutil

# Crear carpeta limpia de artefactos
ARTIFACTS_DIR = OUTPUT_DIR / "embedding_artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

files_to_copy = [
    "item2vec.model",
    "text_embeddings.npy",
    "text_product_ids.npy",
    "text_products.faiss",
    "product_catalog_embeddings.parquet",
    "embedding_examples.json",
    "embedding_latency.json",
    "embedding_report.json"
]

for filename in files_to_copy:
    src = OUTPUT_DIR / filename
    dst = ARTIFACTS_DIR / filename

    if src.exists():
        shutil.copy(src, dst)
        print("Copiado:", filename)
    else:
        print("No encontrado:", filename)

zip_path = shutil.make_archive(
    str(OUTPUT_DIR / "embedding_artifacts"),
    "zip",
    ARTIFACTS_DIR
)

print("ZIP creado:")
print(zip_path)

Copiado: item2vec.model
Copiado: text_embeddings.npy
Copiado: text_product_ids.npy
Copiado: text_products.faiss
Copiado: product_catalog_embeddings.parquet
Copiado: embedding_examples.json
Copiado: embedding_latency.json
Copiado: embedding_report.json
ZIP creado:
/kaggle/working/embedding_artifacts.zip


## Verificación final

Reviso los archivos generados en `/kaggle/working`. Después se debe guardar versión del notebook para que el notebook 04 pueda usar estos outputs.


In [24]:
print("Archivos en /kaggle/working:")

for file in sorted(os.listdir("/kaggle/working")):
    print("-", file)

Archivos en /kaggle/working:
- __notebook__.ipynb
- embedding_artifacts
- embedding_artifacts.zip
- embedding_examples.json
- embedding_latency.json
- embedding_report.json
- item2vec.model
- product_catalog_embeddings.parquet
- text_embeddings.npy
- text_product_ids.npy
- text_products.faiss
